In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

load_dotenv()

agent_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

class State(TypedDict):
    question: str
    message: str
    tool_result: str
    tool_call_count: int
    selected_tool: str
    final_answer: str

def question(state: State):
    question = input("What would you like to do? ")

    return {
        "question": question,
        "message": "",
        "tool_result": "",
        "tool_call_count": 0,
        "selected_tool": "",
        "final_answer": ""
    }

def Agent(state: State):
    question = state["question"]
    tool_result = state["tool_result"]
    selected_tool = state["selected_tool"]

    prompt = f"""
    You are an agent.
    
    Question: {question}
    Previous tool result: {tool_result}
    Previous selected tool: {selected_tool}
    
    Choose exactly ONE tool:
    
    Calculator
    Converter
    String_Analyzer
    Writer
    
    If the previous selected tool was String_Analyzer, choose Writer.
    Do not call String_Analyzer again.
    
    If the previous tool result already contains the final answer
    to the user's request, choose Writer.
    
    When choosing String_Analyzer, if the user wants to analyze
    the previous tool result, pass the complete previous tool result
    as the instruction. Do not remove units, words, or other parts
    of the result.
    
    Only choose another tool if the previous tool result is not
    sufficient to complete the user's request.
    
    Do not repeat a tool that has already completed its required operation.
    
    If String_Analyzer has already been used and returned a numeric
    result, choose Writer. Do not call String_Analyzer again
    for the same request.
    
    If a tool returns that the requested operation is not supported,
    do not try another tool to perform that operation. Choose Writer.
    
    Your response MUST contain exactly TWO lines and nothing else.
    
    Line 1: only the tool name.
    Line 2: only the instruction for that tool.
    
    Example:
    Calculator
    25*4
    
    Do NOT write "Tool:"
    Do NOT write "Instruction:"
    Do NOT add explanations.
"""

    agent_response = agent_llm.invoke(prompt)
    response_text = agent_response.content[0]["text"]

    parts = response_text.split("\n")

    selected_tool = parts[0].strip()
    message = parts[1].strip()

    return {
        "selected_tool": selected_tool,
        "message": message
    }

def Calculator(state: State):
    question = state["message"]

    try:
        result = eval(question)

        return {
            "tool_result": str(result),
            "tool_call_count": state["tool_call_count"] + 1
        }

    except:
        return {
            "tool_result": "Please Give A Valid Question.",
            "tool_call_count": state["tool_call_count"] + 1
        }

def Converter(state: State):
    message = state["message"]
    old_result = state["tool_result"]

    prompt = f"""
    We are using message: {message}, and tool_result: {old_result},
    based on the requirements.
    
    If the tool_result is a length measurement, use these conversions:
    
    meter ↔ kilometer
    centimeter ↔ meter
    millimeter ↔ meter
    meter ↔ centimeter
    kilometer ↔ meter
    
    If the tool_result is Weight/Mass, use these conversions:
    
    gram ↔ kilogram
    milligram ↔ gram
    kilogram ↔ gram
    kilogram ↔ tonne
    tonne ↔ kilogram
    
    If the tool_result is Temperature, use these conversions:
    
    Celsius ↔ Fahrenheit
    Celsius ↔ Kelvin
    Fahrenheit ↔ Celsius
    Kelvin ↔ Celsius
    
    If the tool_result is Time, use these conversions:
    
    seconds ↔ minutes
    minutes ↔ hours
    hours ↔ days
    days ↔ weeks
    
    If the tool_result is Data/Digital Storage, use these conversions:
    
    bytes ↔ KB
    KB ↔ MB
    MB ↔ GB
    GB ↔ TB
    
    If the tool_result is speed, use these conversions:
    
    km/h ↔ m/s
    m/s ↔ km/h
    
    Extract only:
    
    Line 1: numeric value
    Line 2: source unit
    Line 3: target unit
    
    Do not perform the conversion.
    Do not calculate anything.
    Do not add explanations.
"""

    agent_response = agent_llm.invoke(prompt)
    answer = agent_response.content[0]["text"]

    parts = answer.split("\n")

    value = float(parts[0].strip())
    from_unit = parts[1].strip()
    to_unit = parts[2].strip()

    # Length
    if from_unit == "meters" and to_unit == "kilometers":
        result = value / 1000

    elif from_unit == "kilometers" and to_unit == "meters":
        result = value * 1000

    elif from_unit == "centimeters" and to_unit == "meters":
        result = value / 100

    elif from_unit == "meter" and to_unit == "centimeter":
        result = value * 100

    elif from_unit == "millimeters" and to_unit == "meters":
        result = value / 1000

    elif from_unit == "meters" and to_unit == "millimeters":
        result = value * 1000

    # Weight / Mass
    elif from_unit == "grams" and to_unit == "kilograms":
        result = value / 1000

    elif from_unit == "kilograms" and to_unit == "grams":
        result = value * 1000

    elif from_unit == "milligrams" and to_unit == "grams":
        result = value / 1000

    elif from_unit == "grams" and to_unit == "milligrams":
        result = value * 1000

    elif from_unit == "kilograms" and to_unit == "tonnes":
        result = value / 1000

    elif from_unit == "tonnes" and to_unit == "kilograms":
        result = value * 1000

    # Temperature
    elif from_unit == "Celsius" and to_unit == "Fahrenheit":
        result = (value * 9 / 5) + 32

    elif from_unit == "Fahrenheit" and to_unit == "Celsius":
        result = (value - 32) * 5 / 9

    elif from_unit == "Celsius" and to_unit == "Kelvin":
        result = value + 273.15

    elif from_unit == "Kelvin" and to_unit == "Celsius":
        result = value - 273.15

    # Time
    elif from_unit == "seconds" and to_unit == "minutes":
        result = value / 60

    elif from_unit == "minutes" and to_unit == "seconds":
        result = value * 60

    elif from_unit == "minutes" and to_unit == "hours":
        result = value / 60

    elif from_unit == "hours" and to_unit == "minutes":
        result = value * 60

    elif from_unit == "hours" and to_unit == "days":
        result = value / 24

    elif from_unit == "days" and to_unit == "hours":
        result = value * 24

    elif from_unit == "days" and to_unit == "weeks":
        result = value / 7

    elif from_unit == "weeks" and to_unit == "days":
        result = value * 7

    # Data / Digital Storage
    elif from_unit == "bytes" and to_unit == "KB":
        result = value / 1024

    elif from_unit == "KB" and to_unit == "bytes":
        result = value * 1024

    elif from_unit == "KB" and to_unit == "MB":
        result = value / 1024

    elif from_unit == "MB" and to_unit == "KB":
        result = value * 1024

    elif from_unit == "MB" and to_unit == "GB":
        result = value / 1024

    elif from_unit == "GB" and to_unit == "MB":
        result = value * 1024

    elif from_unit == "GB" and to_unit == "TB":
        result = value / 1024

    elif from_unit == "TB" and to_unit == "GB":
        result = value * 1024

    # Speed
    elif from_unit == "km/h" and to_unit == "m/s":
        result = value / 3.6

    elif from_unit == "m/s" and to_unit == "km/h":
        result = value * 3.6

    else:
        result = None

    if result is None:
        converted_result = (
            f"Conversion from {from_unit} to {to_unit} is not supported."
        )
    else:
        if result.is_integer():
            result = int(result)

        converted_result = f"{result} {to_unit}"

    return {
        "tool_call_count": state["tool_call_count"] + 1,
        "tool_result": converted_result
    }

def String_Analyzer(state: State):
    message = state["message"]

    prompt = f"""
    You are a String Analyzer Assistant.
    
    The following is the exact text that must be analyzed:
    
    {message}
    
    Extract the text that should be analyzed.
    
    If the input is already the text to analyze,
    return it exactly as it is.
    
    Return ONLY the extracted text.
    
    Do not count characters.
    Do not add quotes.
    Do not add explanations.
    Do not ask for more information.
"""

    agent_response = agent_llm.invoke(prompt)

    # print("STRING ANALYZER RECEIVED:", message)
    # print("STRING EXTRACTED:", agent_response.text)
    
    count = len(agent_response.text.strip())

    return {
        "tool_result": str(count),
        "tool_call_count": state["tool_call_count"] + 1
    }

def route(state: State):
    tool_call_count = state["tool_call_count"]
    selected_tool = state["selected_tool"]

    if tool_call_count >= 3:
        return "Writer"

    if selected_tool == "Calculator":
        return "Calculator"

    elif selected_tool == "Converter":
        return "Converter"

    elif selected_tool == "String_Analyzer":
        return "String_Analyzer"

    elif selected_tool == "Writer":
        return "Writer"

def Writer(state: State):
    question = state["question"]
    tool_result = state["tool_result"]

    prompt = f"""
    You are the final answer writer.
    
    Original user question:
    {question}
    
    Tool result:
    {tool_result}
    
    Your job is to give the final answer using ONLY the tool result.
    
    Rules:
    
    - Do not perform any calculation.
    - Do not convert units yourself.
    - Do not change, reinterpret, or override the tool result.
    - If the tool result is a character count, report it as a character count.
    - If the tool result is a conversion result, report the conversion result.
    - If the tool result says an operation is not supported,
      clearly report that.
    - Do not invent units or values.
    - Whatever answer you get make sure it is in format as X characters.
"""

    agent_response = agent_llm.invoke(prompt)
    answer = agent_response.text

    return {
        "final_answer": answer
    }

graph = StateGraph(State)

graph.add_node("question", question)
graph.add_node("Agent", Agent)
graph.add_node("Calculator", Calculator)
graph.add_node("Converter", Converter)
graph.add_node("String_Analyzer", String_Analyzer)
graph.add_node("Writer", Writer)

graph.add_edge(START, "question")
graph.add_edge("question", "Agent")

graph.add_conditional_edges("Agent", route)

graph.add_edge("Calculator", "Agent")
graph.add_edge("Converter", "Agent")
graph.add_edge("String_Analyzer", "Agent")

graph.add_edge("Writer", END)

app = graph.compile()

result = app.invoke({})

print(result["final_answer"])

What would you like to do?   Calculate 125557 * 875, then convert the result from meters to centimeters, and finally count the characters in the converted result.


11 characters
